# 03 — Estratégias de Retrieval

## Por que retrieval importa tanto?

Uma resposta de RAG só pode ser tão boa quanto os documentos recuperados.
Se o retrieval falha em encontrar o chunk relevante, o LLM não tem como responder — não importa quão bom seja o modelo.

**Falhas comuns do retrieval simples (dense only):**
- Query usa termos técnicos exatos que o modelo de embedding não captura bem
- Documentos usam sinônimos ou abreviações que o embedding aproxima mal
- Queries muito curtas ("erro 404") têm embeddings pouco informativos
- Keywords raras ou nomes próprios específicos

**A solução:** combinar tipos de retrieval para cobrir pontos cegos de cada um.

| Estratégia | Captura | Falha em |
|-----------|---------|----------|
| Dense (embedding) | Semântica, sinônimos | Keywords exatas, termos raros |
| Sparse (BM25) | Keywords exatas | Sinônimos, contexto |
| **Hybrid (RRF)** | **Ambos** | **Quase nada — padrão de produção** |
| MMR | Relevância + diversidade | Quando você quer só os mais similares |

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import defaultdict
import math

model = SentenceTransformer("all-MiniLM-L6-v2")

# Base de documentos para demonstração
documentos = [
    "HNSW é um algoritmo de indexação vetorial que usa grafos hierárquicos para busca eficiente",
    "O algoritmo Hierarchical Navigable Small World permite busca aproximada em O(log N)",
    "Embeddings são vetores densos que representam o significado semântico de textos",
    "A similaridade de cosseno mede o ângulo entre dois vetores de embedding",
    "RAG combina retrieval de documentos com geração de texto para responder perguntas",
    "Chunking divide documentos longos em pedaços menores para indexação vetorial",
    "BM25 é um algoritmo de ranking baseado em frequência de termos para busca textual",
    "Qdrant é um banco de dados vetorial open-source otimizado para busca semântica",
    "Quantização reduz o uso de memória dos vetores com perda mínima de qualidade",
    "Retrieval-Augmented Generation melhora LLMs com conhecimento externo atualizado",
]

# Pré-computar embeddings dos documentos
doc_embs = model.encode(documentos, normalize_embeddings=True)
print(f"Base: {len(documentos)} documentos indexados")
print(f"Dimensão dos embeddings: {doc_embs.shape[1]}d")

## 3.1 Dense Retrieval (Embedding-based)

O retrieval denso é o que você já conhece: embedding da query → similaridade cosine → top-K.

**Ponto forte:** entende *semântica*. "cachorro" e "cão" são similares.
"como resolver problema X" encontra documentos sobre "soluções para X" sem usar exatamente essas palavras.

**Ponto fraco:** para queries muito específicas com termos técnicos exatos (nomes de erro, siglas, nomes próprios incomuns),
o embedding pode não capturar a especificidade necessária.

**Também falha com negação:** "o que NÃO é machine learning?" — o embedding de
"NÃO machine learning" fica próximo de embeddings de ML, porque o modelo entende o conceito, não a negação.

In [ ]:
def dense_retrieve(query, top_k=3):
    q_emb = model.encode(query, normalize_embeddings=True)
    scores = doc_embs @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(i, documentos[i], float(scores[i])) for i in top_indices]

# Teste com diferentes tipos de query
test_queries = [
    "como funciona a busca em grafos hierárquicos",  # semântica — dense deve ir bem
    "BM25 TF-IDF",                                   # keyword exata — dense pode falhar
    "RAG",                                            # sigla — pode ser ambíguo
]

print("=== DENSE RETRIEVAL ===")
for query in test_queries:
    results = dense_retrieve(query, top_k=2)
    print(f"\nQuery: '{query}'")
    for rank, (idx, doc, score) in enumerate(results, 1):
        print(f"  #{rank} (score={score:.3f}): {doc[:70]}...")

### Análise dos resultados

Para a query semântica ("busca em grafos hierárquicos"), o dense provavelmente encontrou documentos sobre HNSW — mesmo sem essas palavras na query.

Para "BM25 TF-IDF" — uma keyword muito específica — o dense pode não ter performado tão bem, porque o embedding de "BM25 TF-IDF" pode ser próximo de documentos genéricos sobre busca.

Isso motiva o BM25.

## 3.2 Sparse Retrieval (BM25)

BM25 é uma versão melhorada do TF-IDF — o algoritmo clássico de busca textual.

**Como funciona:** pondera palavras por:
- **TF (Term Frequency):** quantas vezes a palavra aparece no documento
- **IDF (Inverse Document Frequency):** palavras raras recebem mais peso que palavras comuns
- **Normalização por tamanho:** documentos longos não ganham vantagem injusta

**Ponto forte:** encontra documentos que contêm exatamente as palavras da query.
Para queries como "CUDA out of memory error", BM25 vai direto nos documentos com essas palavras.

**Ponto fraco:** completamente cego a semântica.
"cachorro" e "cão" são palavras totalmente diferentes para o BM25.

In [ ]:
class BM25:
    """Implementação simples de BM25 para demonstração."""
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = docs
        self.tokenized = [doc.lower().split() for doc in docs]
        self.avgdl = sum(len(d) for d in self.tokenized) / len(self.tokenized)

        # Compute IDF
        self.idf = {}
        N = len(docs)
        for doc in self.tokenized:
            for word in set(doc):
                self.idf[word] = self.idf.get(word, 0) + 1
        self.idf = {w: math.log((N - df + 0.5) / (df + 0.5) + 1)
                    for w, df in self.idf.items()}

    def score(self, query, doc_idx):
        query_tokens = query.lower().split()
        doc = self.tokenized[doc_idx]
        dl = len(doc)
        score = 0
        tf_map = defaultdict(int)
        for w in doc:
            tf_map[w] += 1
        for term in query_tokens:
            if term not in self.idf:
                continue
            tf = tf_map.get(term, 0)
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += self.idf[term] * numerator / denominator
        return score

    def retrieve(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(len(self.docs))]
        scores.sort(key=lambda x: -x[1])
        return [(idx, self.docs[idx], sc) for idx, sc in scores[:top_k]]

bm25 = BM25(documentos)

print("=== SPARSE RETRIEVAL (BM25) ===")
for query in test_queries:
    results = bm25.retrieve(query, top_k=2)
    print(f"\nQuery: '{query}'")
    for rank, (idx, doc, score) in enumerate(results, 1):
        print(f"  #{rank} (score={score:.3f}): {doc[:70]}...")

### Dense vs Sparse: cada um tem seu ponto cego

Compare os resultados das duas estratégias:

- Para queries semânticas: dense tende a ir melhor (encontra sinônimos e contexto)
- Para keywords específicas ("BM25 TF-IDF"): sparse vai melhor (match exato de termos)

Nenhuma estratégia domina a outra em todos os casos. Isso motiva a combinação das duas.

## 3.3 Hybrid Search com RRF (Reciprocal Rank Fusion)

Hybrid search combina os rankings de dense e sparse para cobrir os pontos cegos de cada um.

**RRF** é o algoritmo de fusão mais popular:

```
score_rrf(doc) = 1/(k + rank_dense) + 1/(k + rank_sparse)
```

Onde `k=60` é uma constante empírica. Um documento que aparece bem nos dois rankings tem score máximo.
Um documento que aparece só em um ainda contribui com algum score.

**Por que não somar os scores diretamente?**
Scores de sistemas diferentes têm escalas incompatíveis (cosine: 0-1, BM25: 0-∞).
RRF usa apenas as *posições* (ranks), não os valores absolutos — isso é escala-invariante.

In [ ]:
def hybrid_retrieve(query, top_k=3, k_rrf=60, alpha=0.5):
    """
    Hybrid search com RRF.
    alpha=1.0 -> só dense, alpha=0.0 -> só sparse, alpha=0.5 -> equilíbrio
    """
    # Dense rankings
    q_emb = model.encode(query, normalize_embeddings=True)
    dense_scores = doc_embs @ q_emb
    dense_ranking = {idx: rank+1 for rank, idx in enumerate(np.argsort(dense_scores)[::-1])}

    # Sparse rankings
    bm25_scores = [(i, bm25.score(query, i)) for i in range(len(documentos))]
    bm25_scores.sort(key=lambda x: -x[1])
    sparse_ranking = {idx: rank+1 for rank, (idx, _) in enumerate(bm25_scores)}

    # RRF fusion
    rrf_scores = {}
    for doc_idx in range(len(documentos)):
        dense_rrf = alpha / (k_rrf + dense_ranking.get(doc_idx, len(documentos)))
        sparse_rrf = (1-alpha) / (k_rrf + sparse_ranking.get(doc_idx, len(documentos)))
        rrf_scores[doc_idx] = dense_rrf + sparse_rrf

    top = sorted(rrf_scores.items(), key=lambda x: -x[1])[:top_k]
    return [(idx, documentos[idx], score) for idx, score in top]

print("=== HYBRID SEARCH (RRF, alpha=0.5) ===")
for query in test_queries:
    results = hybrid_retrieve(query, top_k=2)
    print(f"\nQuery: '{query}'")
    for rank, (idx, doc, score) in enumerate(results, 1):
        print(f"  #{rank} (rrf={score:.4f}): {doc[:70]}...")

### Comparação lado a lado

Vamos comparar as 3 estratégias nas mesmas queries para ver onde cada uma vai bem e onde falha:

In [ ]:
print("=== COMPARAÇÃO FINAL ===")
print(f"{'Query':<45} {'Dense':^20} {'Sparse':^20} {'Hybrid':^20}")
print("-" * 105)

for query in test_queries:
    d = dense_retrieve(query, 1)[0][1][:35]
    s = bm25.retrieve(query, 1)[0][1][:35]
    h = hybrid_retrieve(query, 1)[0][1][:35]
    print(f"{query:<45} {d:<20} {s:<20} {h:<20}")

### O que a comparação mostra?

O Hybrid (RRF) tende a ser mais robusto: ele "herda" os acertos de cada estratégia e raramente falha em ambas ao mesmo tempo.

**Regra prática:**
- Hybrid com alpha=0.5 é um excelente padrão para começar
- Se suas queries são muito técnicas com keywords específicas: alpha=0.3 (mais peso no sparse)
- Se suas queries são linguagem natural: alpha=0.7 (mais peso no dense)

## 3.4 MMR (Max Marginal Relevance)

Um problema sutil do retrieval padrão: os top-K resultados podem ser muito similares entre si.

Se você busca "como funciona gradient descent?" e os 5 chunks mais relevantes são parágrafos consecutivos
do mesmo artigo, você está desperdiçando espaço de contexto com informação redundante.

**MMR** resolve isso balanceando **relevância** com **diversidade**:

```
MMR = argmax[λ × sim(doc, query) - (1-λ) × max(sim(doc, já_selecionados))]
```

- λ = 1.0: só relevância (igual ao retrieval padrão)
- λ = 0.5: equilíbrio — bom para cobrir múltiplos ângulos de uma questão

In [ ]:
def mmr_retrieve(query, top_k=3, lambda_param=0.5):
    """Max Marginal Relevance: balanceia relevância com diversidade."""
    q_emb = model.encode(query, normalize_embeddings=True)
    query_scores = doc_embs @ q_emb

    selected = []
    candidates = list(range(len(documentos)))

    while len(selected) < top_k and candidates:
        best_idx = None
        best_score = -np.inf

        for idx in candidates:
            relevance = float(query_scores[idx])
            # Penaliza por similaridade com docs já selecionados
            if selected:
                sim_to_selected = max(float(doc_embs[idx] @ doc_embs[s]) for s in selected)
            else:
                sim_to_selected = 0

            mmr_score = lambda_param * relevance - (1 - lambda_param) * sim_to_selected
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx

        selected.append(best_idx)
        candidates.remove(best_idx)

    return [(idx, documentos[idx], float(query_scores[idx])) for idx in selected]

query = "como funciona a busca vetorial"
print("=== MMR vs DENSE para: '{}' ===".format(query))
print("\nDense (pode ter redundância):")
for rank, (idx, doc, score) in enumerate(dense_retrieve(query, 4), 1):
    print(f"  #{rank}: {doc}")

print("\nMMR (mais diverso):")
for rank, (idx, doc, score) in enumerate(mmr_retrieve(query, 4), 1):
    print(f"  #{rank}: {doc}")

## Resumo

| Estratégia | Recall semântico | Recall keyword | Diversidade | Complexidade |
|-----------|:---:|:---:|:---:|:---:|
| Dense | ★★★★★ | ★★☆☆☆ | ★★☆☆☆ | Baixa |
| Sparse (BM25) | ★★☆☆☆ | ★★★★★ | ★★☆☆☆ | Baixa |
| **Hybrid RRF** | **★★★★★** | **★★★★★** | **★★☆☆☆** | **Média** |
| MMR | ★★★★☆ | ★★☆☆☆ | ★★★★★ | Média |

**Para o seu sistema RAG:**
- Comece com **Hybrid RRF (alpha=0.5)** — cobre os pontos cegos de cada estratégia
- Use **MMR** quando suas queries pedem cobertura de múltiplos aspectos (sumarização, Q&A amplo)
- Use **Dense puro** apenas quando seus documentos são todos linguagem natural sem keywords técnicas

**Próximos passos:**
- [04 — Generation Prompts](04_generation_prompts.html): com os documentos certos em mãos, como pedir ao LLM para gerar a melhor resposta?